# BTCUSDT Microstructure One-Click Collector
Run all cells. The notebook downloads public BTCUSDT futures research data, builds a canonical 5m table, audits gaps, and zips the result.

In [ ]:
!pip -q install -U pandas pyarrow requests tqdm python-dateutil huggingface_hub


In [ ]:
import os, subprocess, textwrap, sys
print('Runtime ready. Internet access is required in Colab.')


In [ ]:
from pathlib import Path
# The collector script is embedded here so the notebook is fully self-contained.
collector = r'''import os, requests, zipfile, json, time, math
from pathlib import Path
import pandas as pd
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed
START=pd.Timestamp(os.environ.get('START_DATE','2025-01-01'),tz='UTC'); END=pd.Timestamp(os.environ.get('END_DATE','2026-05-31'),tz='UTC')
OUT=Path('/content/btcusdt_research'); CACHE=OUT/'cache'; RAW=OUT/'raw'; CAN=OUT/'canonical'; REP=OUT/'reports'
for p in (CACHE,RAW,CAN,REP): p.mkdir(parents=True,exist_ok=True)
BASE='https://data.binance.vision/data/futures/um'; SYMBOL='BTCUSDT'
def months(a,b):
 x=a.normalize().replace(day=1); r=[]
 while x<=b: r.append(x); x=x+pd.offsets.MonthBegin(1)
 return r
def dl(url,path):
 path=Path(path); path.parent.mkdir(parents=True,exist_ok=True); part=Path(str(path)+'.part')
 for k in range(6):
  try:
   n=part.stat().st_size if part.exists() else 0; h={'Range':f'bytes={n}-'} if n else {}
   with requests.get(url,headers=h,stream=True,timeout=90) as r:
    if r.status_code==416: part.replace(path); return
    r.raise_for_status()
    with open(part,'ab' if n else 'wb') as f:
     for c in r.iter_content(1024*1024):
      if c:f.write(c)
   part.replace(path); return
  except Exception as e: print('retry',url,e); time.sleep(2**k)
 raise RuntimeError(url)
def job(t,iv,m):
 s=f'{m.year:04d}-{m.month:02d}'
 if t=='klines': url=f'{BASE}/monthly/klines/{SYMBOL}/{iv}/{SYMBOL}-{iv}-{s}.zip'; p=CACHE/'klines'/iv/f'{s}.zip'
 elif t=='aggTrades': url=f'{BASE}/monthly/aggTrades/{SYMBOL}/{SYMBOL}-aggTrades-{s}.zip'; p=CACHE/'aggTrades'/f'{s}.zip'
 else: url=f'{BASE}/monthly/metrics/{SYMBOL}/{SYMBOL}-metrics-{s}.zip'; p=CACHE/'metrics'/f'{s}.zip'
 try: dl(url,p); return {'type':t,'interval':iv,'month':s,'url':url,'path':str(p),'ok':True}
 except Exception as e: return {'type':t,'interval':iv,'month':s,'url':url,'error':str(e),'ok':False}
jobs=[('klines',i,m) for i in ('1m','5m') for m in months(START,END)] + [(t,None,m) for t in ('aggTrades','metrics') for m in months(START,END)]
with ThreadPoolExecutor(max_workers=8) as ex: res=[f.result() for f in as_completed([ex.submit(job,*j) for j in jobs])]
json.dump(res,open(REP/'download_manifest.json','w'),indent=2)
for r in res: print(r['ok'],r['type'],r['month'])
for z in CACHE.rglob('*.zip'):
 d=RAW/z.parent.relative_to(CACHE); d.mkdir(parents=True,exist_ok=True); marker=d/(z.stem+'.done')
 if not marker.exists():
  try: zipfile.ZipFile(z).extractall(d); marker.write_text('ok')
  except Exception as e: print('extract error',z,e)
xs=[]
for p in (RAW/'klines').rglob('*.csv'):
 try:
  d=pd.read_csv(p,header=None,usecols=range(12),low_memory=False); d.columns=['open_ms','open','high','low','close','volume','close_ms','quote_volume','trade_count','taker_buy_base','taker_buy_quote','ignore']; d['ts']=pd.to_datetime(d.open_ms,unit='ms',utc=True)
  for c in ['open','high','low','close','volume','quote_volume','taker_buy_base','taker_buy_quote']: d[c]=pd.to_numeric(d[c],errors='coerce')
  xs.append(d[['ts','open','high','low','close','volume','quote_volume','trade_count','taker_buy_base','taker_buy_quote']])
 except: pass
k=pd.concat(xs).drop_duplicates('ts').sort_values('ts'); k=k[(k.ts>=START)&(k.ts<=END)]
if k.empty: raise RuntimeError('No Binance kline archive was downloaded. Check Colab internet access.')
k['delta']=2*k.taker_buy_base-k.volume; k['flow_z_48']=(k.delta-k.delta.rolling(48).mean())/k.delta.rolling(48).std(); k['rv_48']=k.close.pct_change().rolling(48).std()
ms=[]
for p in (RAW/'metrics').rglob('*.csv'):
 try:
  d=pd.read_csv(p); d.columns=[str(c).strip() for c in d.columns]
  tc=next((c for c in d.columns if c.lower()=='create_time'),None)
  if tc: d['ts']=pd.to_datetime(d[tc],unit='ms',utc=True)
  ren={}
  for c in d.columns:
   q=c.lower()
   if q=='sum_open_interest': ren[c]='oi'
   elif 'count_toptrader_long_short_ratio' in q: ren[c]='top_ls_count'
   elif 'sum_toptrader_long_short_ratio' in q: ren[c]='top_ls_position'
   elif 'count_long_short_ratio' in q: ren[c]='global_ls_count'
   elif 'sum_taker_long_short_vol_ratio' in q: ren[c]='taker_ls'
  d=d.rename(columns=ren)
  if 'ts' in d: ms.append(d[['ts']+[c for c in ['oi','top_ls_count','top_ls_position','global_ls_count','taker_ls'] if c in d]])
 except: pass
if ms:
 m=pd.concat(ms).drop_duplicates('ts').sort_values('ts'); m=m[(m.ts>=START)&(m.ts<=END)]; k=k.merge(m,on='ts',how='left')
 if 'oi' in k:
  for lag in [1,12,48]: k[f'oi_change_{lag*5}m']=k.oi.diff(lag)
ags=[]
for p in (RAW/'aggTrades').rglob('*.csv'):
 try:
  d=pd.read_csv(p,header=None,usecols=range(7),low_memory=False); d.columns=['id','price','qty','first_id','last_id','ms','buyer_maker']; d['ts']=pd.to_datetime(d.ms,unit='ms',utc=True); d.price=pd.to_numeric(d.price,errors='coerce'); d.qty=pd.to_numeric(d.qty,errors='coerce'); d['signed_qty']=d.qty.where(~d.buyer_maker,-d.qty); d['bucket']=d.ts.dt.floor('5min'); ags.append(d[['id','qty','signed_qty','bucket']])
 except: pass
if ags:
 a=pd.concat(ags).drop_duplicates('id'); a=a.groupby('bucket').agg(agg_delta=('signed_qty','sum'),agg_trades=('id','size'),avg_trade_size=('qty','mean')).reset_index().rename(columns={'bucket':'ts'}); k=k.merge(a,on='ts',how='left')
k['fwd_ret_5m']=k.close.shift(-1)/k.close-1; k['fwd_ret_15m']=k.close.shift(-3)/k.close-1; k['fwd_ret_60m']=k.close.shift(-12)/k.close-1
k.to_parquet(CAN/'btcusdt_futures_5m.parquet',index=False,compression='zstd')
s=k.ts.drop_duplicates(); gaps=s.diff().dt.total_seconds().div(60); g=pd.DataFrame({'from_ts':s.iloc[:-1].to_numpy(),'to_ts':s.iloc[1:].to_numpy(),'gap_minutes':gaps.iloc[1:].to_numpy()}); g=g[g.gap_minutes>5]; g.to_csv(REP/'gaps.csv',index=False)
quality={'rows':int(len(k)),'start':str(k.ts.min()),'end':str(k.ts.max()),'duplicates':int(k.ts.duplicated().sum()),'gap_count':int(len(g)),'null_close':int(k.close.isna().sum())}; json.dump(quality,open(REP/'quality.json','w'),indent=2); print('QUALITY',quality); print('CANONICAL',CAN/'btcusdt_futures_5m.parquet')'''
Path('/content/collector.py').write_text(collector)
!python /content/collector.py


In [ ]:
import zipfile, os
pkg='/content/btcusdt_microstructure_research_package.zip'
with zipfile.ZipFile(pkg,'w',zipfile.ZIP_DEFLATED) as z:
  for p in Path('/content/btcusdt_research').rglob('*'):
    if p.is_file(): z.write(p,p.relative_to('/content/btcusdt_research'))
print('PACKAGE',pkg,round(os.path.getsize(pkg)/1024/1024,2),'MB')
from google.colab import files
files.download(pkg)
